<a href="https://colab.research.google.com/github/miaflynn/CYPLAN255-Final-Project/blob/main/02b_processing_w_tracts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np

#added more that we use in lab
import os
%matplotlib inline
import matplotlib.pyplot as plt
from shapely.geometry import LineString


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!ls /content/drive
!ls /content/drive/Shareddrives

MyDrive  Shareddrives


In [4]:


gdf = pd.read_csv("/content/drive/MyDrive/C255_final_project/cleaned/rbl_total_cleaned_all_dates.csv")

In [5]:
gdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206872 entries, 0 to 206871
Data columns (total 40 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Unnamed: 0                         206872 non-null  int64  
 1   uniqueid                           206872 non-null  object 
 2   business_account_number            206872 non-null  int64  
 3   location_id                        206872 non-null  object 
 4   ownership_name                     206872 non-null  object 
 5   dba_name                           206683 non-null  object 
 6   street_address                     206872 non-null  object 
 7   city                               206872 non-null  object 
 8   state                              206860 non-null  object 
 9   source_zipcode                     206849 non-null  float64
 10  business_start_date                206872 non-null  object 
 11  business_end_date                  8432

In [6]:
# # adding another plot and importing a shpfile of SF so we can see where the points are in SF

# import matplotlib.pyplot as plt

# # Importing SF geometry
# # URL for 2025 TIGER/Line Place boundaries - info here on
# ## how to use: https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html
url = "https://www2.census.gov/geo/tiger/TIGER2025/PLACE/tl_2025_06_place.zip"

places = gpd.read_file(url)

# Filtered to SF
sf_poly = places[
    (places["NAME"] == "San Francisco") &
    (places["STATEFP"] == "06")   # 06 = California
]


# eproject to same as our gdf
sf_poly = sf_poly.to_crs(epsg=4326)

In [7]:
#I added a shp file with census tracts to our folder

tracts_gdf = gpd.read_file(
    "/content/drive/MyDrive/C255_final_project/cb_2020_06_tract_500k"
)

In [8]:
#checking to make sure all good

tracts_gdf.columns

# simplifying
tracts_gdf = tracts_gdf[["NAME", "NAMELSAD", "STATE_NAME","GEOID", "geometry"]]


In [9]:
# was getting error with "index_right" - so troubleshooted this
tracts_gdf = tracts_gdf.drop(columns=["index_right", ], errors="ignore")
gdf = gdf.drop(columns=["index_right"], errors="ignore")

In [10]:
# Joining the gdf and tract gdf so we can summarize within tracts

gdf = gdf.set_geometry(
    gpd.points_from_xy(gdf.lon, gdf.lat)
)

gdf = gdf.set_crs(epsg=4326)
tracts_gdf = tracts_gdf.to_crs(epsg=4326)

# joining within
business_tracts_gdf = gpd.sjoin(gdf, tracts_gdf, how="left", predicate="within")

In [11]:
business_tracts_gdf.columns.tolist()

cols_to_keep = [
    "uniqueid",
    "dba_name",
    "business_account_number",
    "naics_code",
    "naics_code_description",
    "lic_code",
    "lic_code_description",
    "neighborhoods_analysis_boundaries",
    "year",
    "status",
    "location_start_date",
    "location_end_date",
    "GEOID",
    "geometry",
    "lon", "lat"
]

business_tracts_gdf = business_tracts_gdf[cols_to_keep]

In [12]:
sf_tracts = gpd.clip(tracts_gdf, sf_poly)
business_tracts_gdf

,uniqueid,dba_name,business_account_number,naics_code,naics_code_description,lic_code,lic_code_description,neighborhoods_analysis_boundaries,year,status,location_start_date,location_end_date,GEOID,geometry,lon,lat
0,0183810-01-001-0183810,179 Alhambra St Apartments,183810,5300-5399,Real Estate and Rental and Leasing Services,NaN,NaN,Marina,1988,opened,1988-04-01,2015-06-16,06075012601,POINT (-122.43917 37.80152),-122.439172,37.801521
1,0339974-46-001-0339974,Tower Valet Parking Inc,339974,8100-8139,Certain Services,NaN,NaN,Financial District/South Beach,2013,opened,2013-06-01,NaN,06075061507,POINT (-122.39223 37.79035),-122.392229,37.790352
2,1410027-02-261-1165963,Agua Frisco,1165963,7220-7229,Food Services,NaN,NaN,Treasure Island,2026,opened,2026-02-01,NaN,06075017903,POINT (-122.37379 37.82587),-122.373790,37.825868
3,1239658-12-191-1109070,Openwrench,1109070,5100-5199,Information,NaN,NaN,West of Twin Peaks,2019,opened,2019-12-01,2023-06-30,06075031100,POINT (-122.43779 37.73136),-122.437791,37.731357
4,1055584-02-161-1027028,"Propel Venture Partners Mgmt. Co., LLC",1027028,5210-5239,Financial Services,NaN,NaN,Financial District/South Beach,2016,opened,2016-02-01,2020-09-30,06075061501,POINT (-122.40044 37.7888),-122.400436,37.788795
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206867,1320804-12-221-1141617,Stone Gold Clay LLC,1141617,6100-6299,Private Education and Health Services,NaN,NaN,Outer Richmond,2024,closed,2022-12-29,2024-03-10,06075042601,POINT (-122.48322 37.78559),-122.483219,37.785591
206868,1312156-08-221-1137925,Subin Park,1137925,7210-7219,Accommodations,NaN,NaN,Financial District/South Beach,2023,closed,2022-08-16,2023-05-10,06075061506,POINT (-122.39039 37.78806),-122.390391,37.788055
206869,1004645-08-141-1002906,Pch International Usa Inc,1002906,5400-5499,"Professional, Scientific, and Technical Services",NaN,NaN,Potrero Hill,2021,closed,2014-01-01,2021-10-31,06075022702,POINT (-122.39454 37.76473),-122.394537,37.764729
206870,1022437-03-151-1010847,Xenoform Labs LLC,1010847,7100-7199 7210-7219,Multiple,NaN,NaN,Mission,2026,closed,2015-02-24,2026-02-24,06075020801,POINT (-122.42081 37.75703),-122.420812,37.757034


In [16]:
epc_tracts = gpd.read_file("/content/drive/MyDrive/C255_final_project/epc_tracts.geojson")
epc_tracts_sf = gpd.clip(epc_tracts, sf_poly)
epc_tracts_sf.to_parquet('/content/drive/MyDrive/C255_final_project/cleaned/epc_tracts_sf.parquet')
sf_poly.to_parquet('/content/drive/MyDrive/C255_final_project/cleaned/sf_poly.parquet')

In [ ]:
business_tracts_gdf.to_parquet('/content/drive/MyDrive/C255_final_project/cleaned/rbl_GEOID_all_dates.parquet')
sf_tracts.to_parquet('/content/drive/MyDrive/C255_final_project/cleaned/sf_tracts_GEOID.parquet')

In [ ]:
tracts_rde = business_tracts_gdf[business_tracts_gdf['naics_code'].str.contains('4400-4599|7100-7199|7220-7229', regex=True, na=False)]

tracts_rde.to_parquet('/content/drive/MyDrive/C255_final_project/cleaned/rbl_rde_GEOID_all_dates.parquet')